In [1]:
print("hello")

hello


In [2]:
import weave

client = weave.init("ip-ai/eval-inv-gen-new-latest-trails-runlim")
calls = client.get_calls(
    filter={
        "op_names": ["weave:///ip-ai/eval-inv-gen-new-latest-trails-runlim/op/Evaluation.evaluate:*"]
    },
    sort_by=[{"field": "summary.weave.trace_name", "direction": "asc"}]
)

weave: wandb version 0.24.0 is available!  To upgrade, please run:
weave:  $ pip install wandb --upgrade
weave: weave version 0.52.25 is available!  To upgrade, please run:
weave:  $ pip install weave --upgrade
weave: Logged in as Weights & Biases user: ido-pinto.
weave: View Weave data at https://wandb.ai/ip-ai/eval-inv-gen-new-latest-trails-runlim/weave


In [34]:
import pandas as pd
data = []
for call in calls:
  solvable = {
      "display_name": call._display_name,
      "timeout_solved": 0,
      "total": 0,
      "rate": 0.0,
      "vbs_scores": [],
      "vbs_e2e_scores": [],
      'speedup_vals': [],
      'speedup_vals_e2e': [],
      'vbp':0,
      'vbp_e2e':0,
      'avg_speedup':0,
      'avg_speedup_e2e':0,
      'avg_speedup_all':0,
      'avg_speedup_all_e2e':0
  }
  print(call._display_name)
  # Retrieve the children (the individual traces/rows of the evaluation)
  traces = call.children()
  speedups = []
  timeout_time = 600  # Timeout time in seconds
  vbp_scores = []  # VBP scores for all timeout instances
  vbp_e2e_scores = []  # VBP E2E scores for all timeout instances
  all_speedup_vals = []  # Speedup values for all timeout instances (1.0 if no speedup)
  all_speedup_vals_e2e = []  # Speedup E2E values for all timeout instances (1.0 if no speedup)
  
  for trace in traces:
    example_data = trace.inputs.get("example", {})
    if not example_data or not trace.output:
      break
    # 2. Get the scores from the output
    scores= trace.output.get("scores", {})
    scorer_output = scores.get('InvGenScorer', {})
    usefulness_score = scorer_output.get("usefulness_score", 0)
    speedup_val = scorer_output.get('speedup', 0)
    speedup_val_e2e = scorer_output.get('speedup_e2e', 0)
    vbs_score = scorer_output.get('vbs', 0)
    vbs_e2e_score = scorer_output.get('vbs_e2e', 0)
    # 3. Perform the check
    is_timeout = example_data.get('baseline_decision') == 'TIMEOUT'
    has_speedup = speedup_val > 1
    if is_timeout:
      solvable["total"] += 1
      # For VBP: use VBS if speedup, otherwise use timeout time (600)
      if usefulness_score and has_speedup:
        # print(speedup_val)
        solvable["timeout_solved"] += 1
        solvable["vbs_scores"].append(vbs_score)
        solvable["vbs_e2e_scores"].append(vbs_e2e_score)
        solvable["speedup_vals"].append(speedup_val)
        solvable["speedup_vals_e2e"].append(speedup_val_e2e)
        # VBP contribution: use VBS score
        vbp_scores.append(vbs_score)
        vbp_e2e_scores.append(vbs_e2e_score)
        # Speedup contribution: use actual speedup value
        all_speedup_vals.append(speedup_val)
        all_speedup_vals_e2e.append(speedup_val_e2e)
      else:
        # No speedup: use timeout time (600) for VBP, 1.0 for speedup
        vbp_scores.append(timeout_time)
        vbp_e2e_scores.append(timeout_time)
        all_speedup_vals.append(1.0)
        all_speedup_vals_e2e.append(1.0)
  
  solvable["rate"] = solvable["timeout_solved"] / solvable["total"] if solvable["total"] > 0 else 0
  # VBP: mean of all timeout instances (VBS if speedup, 600 if no speedup)
  solvable["vbp"] = sum(vbp_scores) / len(vbp_scores) if len(vbp_scores) > 0 else timeout_time
  solvable["vbp_e2e"] = sum(vbp_e2e_scores) / len(vbp_e2e_scores) if len(vbp_e2e_scores) > 0 else timeout_time
  # Avg speedup: mean over instances with speedup only
  solvable["avg_speedup"] = sum(solvable["speedup_vals"]) / len(solvable["speedup_vals"]) if solvable["speedup_vals"] else 0
  solvable["avg_speedup_e2e"] = sum(solvable["speedup_vals_e2e"]) / len(solvable["speedup_vals_e2e"]) if solvable["speedup_vals_e2e"] else 0
  # Avg speedup all: mean over all timeout instances (actual speedup if speedup, 1.0 if no speedup)
  solvable["avg_speedup_all"] = sum(all_speedup_vals) / len(all_speedup_vals) if len(all_speedup_vals) > 0 else 1.0
  solvable["avg_speedup_all_e2e"] = sum(all_speedup_vals_e2e) / len(all_speedup_vals_e2e) if len(all_speedup_vals_e2e) > 0 else 1.0
  data.append(solvable)

# 1. Create the DataFrame
df = pd.DataFrame(data)

# 2. Sort by rate or total if desired
df = df.sort_values(by="rate", ascending=False)

# 3. Print the DataFrame to the console
print("\n--- Evaluation Summary: Timeout Recovery ---")
print(df.to_string(index=False))

# 4. Save to CSV
output_filename = "timeout_recovery_stats.csv"
df.to_csv(output_filename, index=False)

print(f"\nResults successfully saved to {output_filename}")

eval-Qwen3-0.6B-nt-run0
eval-Qwen3-0.6B-nt-run1
eval-Qwen3-0.6B-nt-run2
eval-Qwen3-0.6B-t-run0
eval-Qwen3-0.6B-t-run1
eval-Qwen3-0.6B-t-run2
eval-Qwen3-14B-nt-run0
eval-Qwen3-14B-nt-run1
eval-Qwen3-14B-nt-run2
eval-Qwen3-4B-Instruct-2507-nt-run0
eval-Qwen3-4B-Instruct-2507-nt-run1
eval-Qwen3-4B-Instruct-2507-nt-run2
eval-Qwen3-8B-nt-run0
eval-Qwen3-8B-nt-run1
eval-Qwen3-8B-nt-run2
eval-Qwen3-8B-t-run0
eval-Qwen3-8B-t-run1
eval-gpt-5.2-run0
eval-gpt-5.2-run2
eval-gpt-oss-120b-run0
eval-gpt-oss-120b-run1
eval-gpt-oss-120b-run2
eval-gpt-oss-20b-run0
eval-gpt-oss-20b-run1
eval-gpt-oss-20b-run2
eval-qwen3-0.6b-gen-inv-sft-v0-latest-nt-run0
eval-qwen3-0.6b-gen-inv-sft-v0-latest-nt-run1
eval-qwen3-0.6b-gen-inv-sft-v0-latest-nt-run2
eval-qwen3-0.6b-gen-inv-sft-v1-latest-nt-run0
eval-qwen3-0.6b-gen-inv-sft-v1-latest-nt-run1
eval-qwen3-0.6b-gen-inv-sft-v1-latest-nt-run2
eval-qwen3-0.6b-nt-gen-inv-sft-v2.2-latest-nt-run0
eval-qwen3-0.6b-nt-gen-inv-sft-v2.2-latest-nt-run1
eval-qwen3-0.6b-nt-gen-in

In [36]:
df.head()

,display_name,timeout_solved,total,rate,vbs_scores,vbs_e2e_scores,speedup_vals,speedup_vals_e2e,vbp,vbp_e2e,avg_speedup,avg_speedup_e2e,avg_speedup_all,avg_speedup_all_e2e
39,eval-qwen3-1.7b-nt-gen-inv-sft-v2.2-latest-nt-...,5,20,0.250000,"[49.32, 9.19, 45.5, 7.32, 305.18]","[49.79310735506471, 9.643345197020098, 45.8943...","[12.187956204379562, 65.40914036996736, 13.211...","[12.072152792425758, 62.33417841204624, 13.097...",470.825500,470.936679,34.979370,33.479299,9.494842,9.119825
52,eval-qwen3-8b-nt-gen-inv-sft-v2.2-latest-nt-run2,4,20,0.200000,"[31.22, 9.07, 9.32, 384.12]","[31.866197363035752, 9.714281215034426, 10.059...","[19.254003843689944, 66.27453142227122, 64.496...","[18.863562324423352, 61.8789992480025, 59.7551...",501.686500,501.829567,37.897554,35.514810,8.379511,7.902962
16,eval-Qwen3-8B-t-run1,3,19,0.157895,"[43.88, 43.4, 426.02]","[117.42777739499695, 133.31445427702275, 469.8...","[13.698951686417502, 13.850460829493088, 1.410...","[5.118976219553403, 4.508963437309765, 1.27946...",532.278947,543.187215,9.653468,3.635800,2.366337,1.416179
19,eval-gpt-5.2-run2,3,20,0.150000,"[59.09, 43.33, 6.91]","[77.76749337013811, 81.62862020676955, 22.7027...","[10.172787273650364, 13.872836372028619, 86.99...","[7.72957921041621, 7.363961297855544, 26.47743...",515.466500,519.104942,37.012314,13.856992,6.401847,2.928549
14,eval-Qwen3-8B-nt-run2,3,20,0.150000,"[52.27, 7.41, 36.65]","[53.22743688603398, 8.391146563077346, 41.7027...","[11.50009565716472, 81.12145748987854, 16.4013...","[11.293235879214796, 71.6362174681824, 14.4141...",514.816500,515.166067,36.340972,32.447869,6.301146,5.717180


In [37]:
# Group by models and calculate mean and std across runs
import re

# Extract model name from display_name (remove run number)
df['model'] = df['display_name'].str.replace(r'-run\d+$', '', regex=True)

# Group by model and calculate statistics
metrics = ['rate', 'vbp', 'vbp_e2e', 'avg_speedup', 'avg_speedup_e2e', 'avg_speedup_all', 'avg_speedup_all_e2e']

# Calculate mean and std for each metric across runs
# Only include models where all runs have total=20 and exactly 3 runs
max_total = df['total'].max()
grouped_stats = []
for model in df['model'].unique():
    model_data = df[df['model'] == model]
    
    # Check if model has exactly 3 runs
    if len(model_data) != 3:
        continue
    
    # Check if all runs have total equal to max_total (20)
    if not all(model_data['total'] == max_total):
        continue
    
    stats = {'model': model}
    
    # Add count (timeout_solved) mean and std
    timeout_solved_values = model_data['timeout_solved'].values
    if len(timeout_solved_values) > 0:
        stats['count_mean'] = round(timeout_solved_values.mean(), 2)
        stats['count_std'] = round(timeout_solved_values.std(), 2)
    else:
        stats['count_mean'] = 0.0
        stats['count_std'] = 0.0
    
    for metric in metrics:
        values = model_data[metric].values
        if len(values) > 0:
            stats[f'{metric}_mean'] = round(values.mean(), 2)
            stats[f'{metric}_std'] = round(values.std(), 2)
        else:
            stats[f'{metric}_mean'] = 0.0
            stats[f'{metric}_std'] = 0.0
    
    grouped_stats.append(stats)

# Create summary dataframe
summary_df = pd.DataFrame(grouped_stats)

# Sort by rate_mean descending
summary_df = summary_df.sort_values(by='rate_mean', ascending=False)

# Display results
print("\n--- Model Summary: Mean and Std across runs (run0, run1, run2) ---")
print(summary_df.to_string(index=False))

# Save to CSV
output_filename = "timeout_recovery_stats_by_model.csv"
summary_df.to_csv(output_filename, index=False)
print(f"\nResults successfully saved to {output_filename}")


--- Model Summary: Mean and Std across runs (run0, run1, run2) ---
                                        model  count_mean  count_std  rate_mean  rate_std  vbp_mean  vbp_std  vbp_e2e_mean  vbp_e2e_std  avg_speedup_mean  avg_speedup_std  avg_speedup_e2e_mean  avg_speedup_e2e_std  avg_speedup_all_mean  avg_speedup_all_std  avg_speedup_all_e2e_mean  avg_speedup_all_e2e_std
eval-qwen3-1.7b-nt-gen-inv-sft-v2.2-latest-nt        3.33       1.25       0.17      0.06    509.11    29.65        509.18        29.62             52.67            13.85                 49.83                12.79                  9.00                 2.01                      8.57                     1.89
                                 eval-gpt-5.2        2.67       0.47       0.13      0.02    524.29    13.16        526.77        12.72             36.22             3.91                 14.66                 1.35                  5.78                 1.27                      2.84                     0.44
       e

In [40]:
# Generate LaTeX table with count, VBP, and avg_speedup
import re

# Map model names to cleaner display names
def clean_model_name(model_name):
    """Convert model name to display format"""
    # Remove eval- prefix
    name = model_name.replace('eval-', '')
    
    # Map specific models
    name_mapping = {
        'gpt-5.2': 'GPT-5.2',
        'gpt-oss-120b': 'GPT-OSS-120B',
        'gpt-oss-20b': 'GPT-OSS-20B',
        'Qwen3-8B-nt': 'Qwen3-8B (Base)',
        'qwen3-8b-nt-gen-inv-sft-v2.2-latest-nt': 'Qwen3-8B-V2 (Ours)',
        'Qwen3-4B-Instruct-2507-nt': 'Qwen3-4B-Instruct-2507 (Base)',
        'qwen3-4b-nt-gen-inv-sft-v2.2-latest-nt': 'Qwen3-4B-Instruct-2507-V2 (Ours)',
        'Qwen3-0.6B-nt': 'Qwen3-0.6B (Base)',
        'qwen3-0.6b-nt-gen-inv-sft-v2.2-latest-nt': 'Qwen3-0.6B-V2 (Ours)',
    }
    
    # Try exact match first
    if name in name_mapping:
        return name_mapping[name]
    
    # Try case-insensitive match
    for key, value in name_mapping.items():
        if name.lower() == key.lower():
            return value
    
    # Default: capitalize and clean up
    return name.replace('-', ' ').title()

# Define model groups (order matters)
model_groups = [
    ['eval-gpt-5.2', 'eval-gpt-oss-120b', 'eval-gpt-oss-20b'],
    ['eval-Qwen3-8B-nt', 'eval-qwen3-8b-nt-gen-inv-sft-v2.2-latest-nt'],
    ['eval-Qwen3-4B-Instruct-2507-nt', 'eval-qwen3-4b-nt-gen-inv-sft-v2.2-latest-nt'],
    ['eval-Qwen3-0.6B-nt', 'eval-qwen3-0.6b-nt-gen-inv-sft-v2.2-latest-nt'],
]

# Generate LaTeX table
latex_lines = []
latex_lines.append("\\begin{table*}")
latex_lines.append("    \\centering")
latex_lines.append("    \\setlength{\\tabcolsep}{0.9em}")
latex_lines.append("    \\caption{Timeout recovery metrics. Solved Count: number of timeouts solved across 3 runs (out of 20). VBP: Virtual Best Performance (seconds, lower is better), calculated over all timeout instances. $\\bar{S}_{>1}$: average speedup factor over instances with speedup only. $\\bar{S}_{all}$: average speedup factor over all timeout instances (actual speedup if speedup, 1.0 if no speedup).}")
latex_lines.append("    \\begin{tabular}{l | r r r r}")
latex_lines.append("        \\toprule")
latex_lines.append("        Model & Solved Count & VBP $\\downarrow$ & $\\bar{S}_{>1}$ $\\uparrow$ & $\\bar{S}_{all}$ $\\uparrow$ \\\\")
latex_lines.append("        \\midrule")

for group_idx, group in enumerate(model_groups):
    # Find best (lowest) VBP in this group
    group_vbps = []
    for model_name in group:
        if model_name in summary_df['model'].values:
            row = summary_df[summary_df['model'] == model_name].iloc[0]
            group_vbps.append((model_name, row['vbp_mean']))
    
    best_vbp = min(group_vbps, key=lambda x: x[1])[0] if group_vbps else None
    
    for model_name in group:
        if model_name in summary_df['model'].values:
            row = summary_df[summary_df['model'] == model_name].iloc[0]
            display_name = clean_model_name(model_name)
            
            # Get individual run counts from original df, sorted by run number
            model_runs = df[df['model'] == model_name].copy()
            # Extract run number and sort
            def extract_run_num(display_name):
                match = re.search(r'-run(\d+)$', display_name)
                return int(match.group(1)) if match else 999
            model_runs['run_num'] = model_runs['display_name'].apply(extract_run_num)
            model_runs = model_runs.sort_values('run_num')
            run_counts = []
            for _, run_row in model_runs.iterrows():
                run_counts.append(int(run_row['timeout_solved']))
            count_str = f"[{', '.join(map(str, run_counts))}]"
            
            vbp_str = f"{row['vbp_mean']:.1f}$\\pm${row['vbp_std']:.1f}"
            # Bold the best VBP in each group
            if model_name == best_vbp:
                vbp_str = f"\\textbf{{{vbp_str}}}"
            speedup_str = f"{row['avg_speedup_mean']:.1f}$\\pm${row['avg_speedup_std']:.1f}"
            speedup_all_str = f"{row['avg_speedup_all_mean']:.1f}$\\pm${row['avg_speedup_all_std']:.1f}"
            latex_lines.append(f"        {display_name} & {count_str} & {vbp_str} & {speedup_str} & {speedup_all_str} \\\\")
    
    if group_idx < len(model_groups) - 1:
        latex_lines.append("        \\midrule")

latex_lines.append("        \\bottomrule")
latex_lines.append("    \\end{tabular}")
latex_lines.append("    \\label{tab:timeout-recovery}")
latex_lines.append("\\end{table*}")

latex_table = "\n".join(latex_lines)
print(latex_table)

\begin{table*}
    \centering
    \setlength{\tabcolsep}{0.9em}
    \caption{Timeout recovery metrics. Solved Count: number of timeouts solved across 3 runs (out of 20). VBP: Virtual Best Performance (seconds, lower is better), calculated over all timeout instances. $\bar{S}_{>1}$: average speedup factor over instances with speedup only. $\bar{S}_{all}$: average speedup factor over all timeout instances (actual speedup if speedup, 1.0 if no speedup).}
    \begin{tabular}{l | r r r r}
        \toprule
        Model & Solved Count & VBP $\downarrow$ & $\bar{S}_{>1}$ $\uparrow$ & $\bar{S}_{all}$ $\uparrow$ \\
        \midrule
        GPT-5.2 & [3, 2, 3] & \textbf{524.3$\pm$13.2} & 36.2$\pm$3.9 & 5.8$\pm$1.3 \\
        GPT-OSS-120B & [3, 2, 1] & 558.7$\pm$29.6 & 9.5$\pm$9.6 & 2.2$\pm$1.5 \\
        GPT-OSS-20B & [3, 2, 2] & 550.5$\pm$13.3 & 25.4$\pm$15.3 & 3.9$\pm$1.8 \\
        \midrule
        Qwen3-8B (Base) & [0, 0, 3] & 571.6$\pm$40.2 & 12.1$\pm$17.1 & 2.8$\pm$2.5 \\
        Qwen3-8B-